## Systolic array implementation

We wish to multiply `Y = X.dot(W)` where `W` is `nw x ncol` and `X` is `nrow x nw`.

In [45]:
import numpy as np

nrow = 5
ncol = 4
nw = 3

X = np.random.randint(0, 5, (nrow, nw))
W = np.random.randint(0, 5, (nw, ncol))
Y = X @ W



Now we perform the systolic array implementation.

In iteration `t`:

```
  Y[i,j] = \sum_k X[i,k] W[k,j]
```

Define the partial sum:

```
  P[i+s,s,j] = \sum_{k=0}^s X[i,k] W[k,j]
```

so for `s=0`:

```
   P[i,0,j] = X[i,0] * W[0, j]
```

For `s > 0`:

```
P[i+s,s,j] = P[i+s-1,s-1,j] + X[i,s] * W[s,j]
```


In [47]:
nt = nrow + nw - 1
Xdly = np.zeros((nw, nw))

print("X = ")
print(X)
print("W = ")
print(W)
print("Y = ")   
print(Y)

# Partial sums for the systolic array
P = np.zeros((nw, ncol))
Yout = np.zeros((nrow, ncol))

for t in range(nt):
    for k in range(nw):
        for d in range(k, 0, -1):
            Xdly[d, k] = Xdly[d - 1, k]
        if t < nrow:
            Xdly[0, k] = X[t, k]
        else:
            Xdly[0, k] = 0

    for j in range(ncol):
        for k in range(nw - 1, 0, -1):
            x_row = t-k
            if x_row >= 0 and x_row < nrow:
                P[k, j] = P[k - 1, j] + X[x_row, k] * W[k, j]
            else:
                P[k, j] = P[k - 1, j]  
                
        P[0, j] = Xdly[0, 0] * W[0, j]  

    if t >= nw - 1:
        Yout[t - (nw - 1), :] = P[-1, :]

print("Yout =")
print(Yout)
print("Y =")
print(Y)


X = 
[[0 4 3]
 [3 2 0]
 [4 0 0]
 [2 2 4]
 [2 4 0]]
W = 
[[4 2 2 0]
 [3 2 2 4]
 [2 4 0 0]]
Y = 
[[18 20  8 16]
 [18 10 10  8]
 [16  8  8  0]
 [22 24  8  8]
 [20 12 12 16]]
Yout =
[[18. 20.  8. 16.]
 [18. 10. 10.  8.]
 [16.  8.  8.  0.]
 [22. 24.  8.  8.]
 [20. 12. 12. 16.]]
Y =
[[18 20  8 16]
 [18 10 10  8]
 [16  8  8  0]
 [22 24  8  8]
 [20 12 12 16]]


In [54]:
print("X =")
print(X)
print("W=")
print(W)
print("Y =  ")
print(Y)

nt = nrow + 2*nw
Xdly = np.zeros((nw,ncol))
S = np.zeros((nw,ncol)) 
Yout = np.zeros((nrow,ncol))

for t in range(nt):  
    # Shift down the Xdly array
    for d in range(ncol-1, 0, -1):
        Xdly[:,d] = Xdly[:,d-1]

    # Shift in new values from X
    for k in range(nw):        
        if t-k >= 0 and t-k < nrow:
            Xdly[k,0] = X[t-k,k]
        else:
            Xdly[k,0] = 0

    # Compute products
    P = Xdly * W

    # Compute partial sums
    for k in range(nw-1,0,-1):
        S[k,:] = P[k,:] + S[k-1,:]
    S[0,:] = P[0,:]

    # Shift out results to Yout
    for j in range(ncol):
        i = t - nw -j + 1
        if i >= 0 and i < nrow:
            Yout[i,j] = S[-1,j]



    #print(f"t={t}")
    #print(S)

print("Yout =")
print(Yout)

X =
[[0 4 3]
 [3 2 0]
 [4 0 0]
 [2 2 4]
 [2 4 0]]
W=
[[4 2 2 0]
 [3 2 2 4]
 [2 4 0 0]]
Y =  
[[18 20  8 16]
 [18 10 10  8]
 [16  8  8  0]
 [22 24  8  8]
 [20 12 12 16]]
Yout =
[[18. 20.  8. 16.]
 [18. 10. 10.  8.]
 [16.  8.  8.  0.]
 [22. 24.  8.  8.]
 [20. 12. 12. 16.]]
